# GeoSlide — Phase 8: Production Model Export

**Project:** GeoSlide — Wireless Sensor Network (WSN) Landslide Prediction
**Phase:** 8 of N — Production Model Export
**Dataset:** `wsn_landslide_data.csv`

## Scope of this Notebook

This notebook is scoped **strictly to packaging already-trained artifacts for deployment**. It performs the
following steps only:

1. Load the final **KNN prediction model** (Phase 5) — no retraining.
2. Load the **Random Forest explainability model** (Phase 7, used for SHAP) — no retraining.
3. Load the fitted **`StandardScaler`** (Phase 5) — no retraining.
4. Persist the feature name list.
5. Write a `metadata.json` describing the production pipeline.
6. Verify every exported artifact exists on disk.
7. Print a deployment-ready summary.

**No models are trained from scratch in this notebook.** Every model is loaded from the artifact saved by
its originating phase. The **only** exception is a narrow, clearly-flagged fallback: the Phase 7 SHAP
notebook trained the Random Forest explainer in-memory but never persisted it to disk. If — and only if —
`models/rf_explainer.joblib` (or an equivalent already-exported artifact) cannot be found, this notebook
reconstructs that single model using the **exact** specification from Phase 7 (`RandomForestClassifier`,
`n_estimators=200`, `random_state=42`, trained on the Phase-7 80/20 stratified split with `random_state=42`,
fit on `StandardScaler`-transformed features). This is a load-with-fallback pattern, not a retraining /
model-selection step — no hyperparameters are tuned and no new modeling decisions are made.

**Explicitly out of scope for this notebook:**
- Model training or retraining
- Hyperparameter tuning
- Model comparison or selection
- Evaluation metrics
- SHAP analysis


## 1. Environment Setup

Import the libraries needed to load persisted models, and (only as a fallback — see above) the small set of
`scikit-learn` utilities required to faithfully reconstruct the Phase 7 Random Forest explainer if its
artifact is missing.


In [ ]:
import os
import json
import pickle
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib

# Only used in the narrow fallback path described above
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

print("Environment ready.")


## 2. Configuration

Paths and constants mirror the earlier phases exactly, so that any fallback reconstruction is byte-for-byte
consistent with how the artifacts were originally produced.


In [ ]:
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

DATA_PATH = "wsn_landslide_data.csv"

TARGET_COL = "Label"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Source artifacts (from Phase 5 / Phase 7)
KNN_SOURCE_CANDIDATES = [
    os.path.join(MODELS_DIR, "knn_model.joblib"),
    "knn_model.joblib",
]
SCALER_SOURCE_CANDIDATES = [
    os.path.join(MODELS_DIR, "scaler.joblib"),
    "scaler.joblib",
]
RF_EXPLAINER_SOURCE_CANDIDATES = [
    os.path.join(MODELS_DIR, "rf_explainer.joblib"),
    os.path.join(MODELS_DIR, "rf_explainer_model.joblib"),
    "rf_explainer.joblib",
]

# Production export targets (Phase 8 deliverables)
KNN_EXPORT_PATH = os.path.join(MODELS_DIR, "knn_model.joblib")
RF_EXPLAINER_EXPORT_PATH = os.path.join(MODELS_DIR, "rf_explainer.joblib")
SCALER_EXPORT_PATH = os.path.join(MODELS_DIR, "scaler.joblib")
FEATURE_NAMES_EXPORT_PATH = os.path.join(MODELS_DIR, "feature_names.pkl")
METADATA_EXPORT_PATH = "metadata.json"

print("Target column     :", TARGET_COL)
print("Random state       :", RANDOM_STATE)
print("Train/test split   :", f"{int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}", "(stratified)")
print("Models directory   :", os.path.abspath(MODELS_DIR))


## 3. Reproduce the Phase 5 / Phase 7 Train-Test Split (Reference Only)

Both the KNN pipeline (Phase 5) and the Random Forest explainer (Phase 7) were built on the **same** 80/20
stratified split (`random_state=42`) of `wsn_landslide_data.csv`. This split is only recreated here:

- to recover the exact feature name order if it cannot be read off an already-saved artifact, and
- as the fallback path used **only if** a required model/scaler artifact is missing from disk.

If all artifacts are already present on disk, this dataset is not required and loading is skipped entirely.


In [ ]:
def _load_and_split_dataset():
    # Recreate the Phase 5 / Phase 7 preprocessing exactly. Used only as a fallback.
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(
            f"'{DATA_PATH}' was not found. It is required to reconstruct a missing artifact. "
            f"Please place the dataset in the working directory and re-run this cell."
        )

    df = pd.read_csv(DATA_PATH)
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    # Phase 7 defensively drops rows with a missing target before splitting; Phase 5 assumes a
    # clean target column. Applying the same dropna here is a no-op when the data is already clean,
    # and keeps this fallback consistent with both source notebooks.
    valid_idx = y.dropna().index
    X, y = X.loc[valid_idx], y.loc[valid_idx]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test


_dataset_cache = {}

def get_split():
    # Lazily load + cache the train/test split so it is computed at most once.
    if not _dataset_cache:
        X_train, X_test, y_train, y_test = _load_and_split_dataset()
        _dataset_cache["X_train"] = X_train
        _dataset_cache["X_test"] = X_test
        _dataset_cache["y_train"] = y_train
        _dataset_cache["y_test"] = y_test
    return (
        _dataset_cache["X_train"],
        _dataset_cache["X_test"],
        _dataset_cache["y_train"],
        _dataset_cache["y_test"],
    )

print("Split-reconstruction helper ready (only invoked if an artifact is missing).")


## 4. Load the Production Pipeline: KNN Model + Scaler (Phase 5)

Both artifacts are loaded directly from the `models/` directory produced by Phase 5. **No training occurs
here** unless the saved artifacts genuinely cannot be found, in which case they are recreated using the
exact Phase 5 specification (`KNeighborsClassifier()` default hyperparameters, fit on
`StandardScaler`-transformed training data from the split above).


In [ ]:
def _find_existing(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def load_scaler():
    path = _find_existing(SCALER_SOURCE_CANDIDATES)
    if path:
        scaler = joblib.load(path)
        print(f"Loaded fitted StandardScaler from: {path}")
        return scaler

    print("WARNING: No saved scaler.joblib found. Recreating exactly as in Phase 5 "
          "(fit only on X_train).")
    X_train, _, _, _ = get_split()
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def load_knn_model(scaler):
    path = _find_existing(KNN_SOURCE_CANDIDATES)
    if path:
        model = joblib.load(path)
        print(f"Loaded trained KNN model from: {path}")
        return model

    print("WARNING: No saved knn_model.joblib found. Recreating exactly as in Phase 5 "
          "(KNeighborsClassifier(), default hyperparameters, trained on scaled features).")
    from sklearn.neighbors import KNeighborsClassifier
    X_train, _, y_train, _ = get_split()
    X_train_scaled = scaler.transform(X_train)
    model = KNeighborsClassifier()
    model.fit(X_train_scaled, y_train)
    return model


scaler = load_scaler()
knn_model = load_knn_model(scaler)

print("\nPrediction pipeline ready:")
print(" - Model :", type(knn_model).__name__)
print(" - Scaler:", type(scaler).__name__)


## 5. Load the Explainability Model: Random Forest (Phase 7)

The Random Forest explainer trained in Phase 7 was used only in-memory for SHAP and was never persisted.
This notebook first checks whether it (or a previous Phase 8 run's export) already exists on disk; only if
it does not is it reconstructed, using the **exact** Phase 7 specification:
`RandomForestClassifier(n_estimators=200, random_state=42)`, trained on `StandardScaler`-transformed
features from the same 80/20 stratified split (`random_state=42`).

Note this is a **separate** model from the default-hyperparameter Random Forest trained in Phase 5
(`random_forest_model.joblib`), which was never used for SHAP and is not part of this export.


In [ ]:
def load_rf_explainer(scaler):
    path = _find_existing(RF_EXPLAINER_SOURCE_CANDIDATES)
    if path:
        model = joblib.load(path)
        print(f"Loaded Random Forest explainability model from: {path}")
        return model

    print("WARNING: No saved rf_explainer.joblib found (Phase 7 kept this model in-memory only). "
          "Recreating exactly as in Phase 7 "
          "(RandomForestClassifier(n_estimators=200, random_state=42), trained on scaled features).")
    X_train, _, y_train, _ = get_split()
    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=X_train.columns, index=X_train.index
    )
    model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    model.fit(X_train_scaled, y_train)
    return model


rf_explainer_model = load_rf_explainer(scaler)

print("\nExplainability model ready:")
print(" - Model        :", type(rf_explainer_model).__name__)
print(" - n_estimators :", rf_explainer_model.n_estimators)
print(" - random_state :", rf_explainer_model.random_state)


## 6. Resolve Feature Names

Feature names are recovered, in order of preference, from:
1. The fitted scaler's `feature_names_in_` (present whenever it was fit on a `DataFrame`, as in Phase 5/7).
2. The dataset on disk, if step 1 is unavailable and the dataset can be loaded.

This avoids re-reading the dataset when it isn't necessary.


In [ ]:
if hasattr(scaler, "feature_names_in_"):
    feature_names = list(scaler.feature_names_in_)
    print("Resolved feature names from the fitted scaler.")
else:
    X_train, _, _, _ = get_split()
    feature_names = list(X_train.columns)
    print("Resolved feature names from the dataset.")

print(f"Number of features: {len(feature_names)}")
print("Feature names:", feature_names)


## 7. Export Artifacts

Every artifact is written to the `models/` directory in the exact filenames required for production
deployment.


In [ ]:
# 1. Final KNN prediction model
joblib.dump(knn_model, KNN_EXPORT_PATH)
print(f"Exported KNN prediction model -> {KNN_EXPORT_PATH}")

# 2. Random Forest explainability model
joblib.dump(rf_explainer_model, RF_EXPLAINER_EXPORT_PATH)
print(f"Exported Random Forest explainability model -> {RF_EXPLAINER_EXPORT_PATH}")

# 3. Fitted StandardScaler
joblib.dump(scaler, SCALER_EXPORT_PATH)
print(f"Exported fitted StandardScaler -> {SCALER_EXPORT_PATH}")

# 4. Feature names
with open(FEATURE_NAMES_EXPORT_PATH, "wb") as f:
    pickle.dump(feature_names, f)
print(f"Exported feature names -> {FEATURE_NAMES_EXPORT_PATH}")


## 8. Write `metadata.json`

A single metadata file documents the production pipeline: which model predicts, which model explains, the
target column, the reproducibility settings, and the preprocessing used.


In [ ]:
metadata = {
    "project": "GeoSlide - WSN Landslide Prediction",
    "phase": "Phase 8 - Production Model Export",
    "export_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "prediction_model": {
        "name": "K-Nearest Neighbors (KNN)",
        "role": "final production prediction model",
        "file": KNN_EXPORT_PATH,
        "library": "scikit-learn",
        "class": type(knn_model).__name__,
        "hyperparameters": knn_model.get_params(),
        "trained_on": "StandardScaler-transformed features",
    },
    "explainability_model": {
        "name": "Random Forest",
        "role": "SHAP explainability model (TreeExplainer)",
        "file": RF_EXPLAINER_EXPORT_PATH,
        "library": "scikit-learn",
        "class": type(rf_explainer_model).__name__,
        "hyperparameters": {
            "n_estimators": rf_explainer_model.n_estimators,
            "random_state": rf_explainer_model.random_state,
        },
        "trained_on": "StandardScaler-transformed features",
        "note": "Used exclusively for SHAP explanations; does not replace the KNN prediction model.",
    },
    "target_column": TARGET_COL,
    "random_state": RANDOM_STATE,
    "train_test_split": {
        "test_size": TEST_SIZE,
        "train_size": round(1 - TEST_SIZE, 2),
        "stratified": True,
    },
    "scaler": {
        "name": "StandardScaler",
        "file": SCALER_EXPORT_PATH,
        "fit_on": "X_train only (no data leakage)",
    },
    "features": {
        "count": len(feature_names),
        "file": FEATURE_NAMES_EXPORT_PATH,
        "names": feature_names,
    },
}

with open(METADATA_EXPORT_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Exported metadata -> {METADATA_EXPORT_PATH}")
print(json.dumps(metadata, indent=2))


## 9. Verify Every Exported File Exists

A final integrity check confirms every required production artifact was actually written to disk before
declaring the export complete.


In [ ]:
required_files = {
    "KNN prediction model": KNN_EXPORT_PATH,
    "Random Forest explainability model": RF_EXPLAINER_EXPORT_PATH,
    "Fitted StandardScaler": SCALER_EXPORT_PATH,
    "Feature names": FEATURE_NAMES_EXPORT_PATH,
    "Metadata": METADATA_EXPORT_PATH,
}

print("Verification of exported artifacts:")
all_ok = True
for label, path in required_files.items():
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status:7}] {label:38} -> {path}  ({size_kb:,.1f} KB)")

if not all_ok:
    raise RuntimeError("One or more required export files are missing. See the list above.")

print("\nAll required files verified successfully.")


## 10. Deployment-Ready Summary


In [ ]:
print("=" * 60)
print("GeoSlide — Phase 8: Production Model Export — COMPLETE")
print("=" * 60)
print(f"Prediction model       : KNN  ({KNN_EXPORT_PATH})")
print(f"Explainability model   : Random Forest  ({RF_EXPLAINER_EXPORT_PATH})")
print(f"Scaler                 : StandardScaler  ({SCALER_EXPORT_PATH})")
print(f"Feature names          : {len(feature_names)} features  ({FEATURE_NAMES_EXPORT_PATH})")
print(f"Metadata               : {METADATA_EXPORT_PATH}")
print(f"Target column          : {TARGET_COL}")
print(f"Random state            : {RANDOM_STATE}")
print(f"Train/test split        : {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} (stratified)")
print("-" * 60)
print("Status: READY FOR DEPLOYMENT")
print("=" * 60)
